# 🧠 Servidor de Consciencia AI - Google Colab

Este notebook configura y ejecuta el servidor de consciencia AI en Google Colab, haciéndolo accesible desde tu computadora local usando ngrok.

## Características:
- ✅ Instalación automática de dependencias
- ✅ Configuración de OpenAI API
- ✅ Túnel ngrok para acceso remoto
- ✅ Endpoints REST para procesamiento de consciencia
- ✅ Soporte bilingüe (Español/Inglés)

## 📦 Paso 1: Clonar el repositorio y preparar el entorno

In [ ]:
# Clonar el repositorio
!git clone https://github.com/diazvaldiviav/Minimous_Concsience_AI.git
%cd Minimous_Concsience_AI

# Verificar que estamos en el directorio correcto
!pwd
!ls -la

## 📚 Paso 2: Instalar dependencias

In [ ]:
# Instalar dependencias principales
!pip install -q torch numpy transformers
!pip install -q openai python-dotenv
!pip install -q fastapi uvicorn pydantic
!pip install -q pyngrok

# Instalar dependencias adicionales del proyecto
!pip install -q langdetect nltk scikit-learn matplotlib

# Descargar recursos NLTK si es necesario
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

print("✅ Todas las dependencias instaladas correctamente")

## 🔑 Paso 3: Configurar OpenAI API Key

In [ ]:
import os
from getpass import getpass

# Solicitar API key de OpenAI de forma segura
openai_key = getpass("🔑 Ingresa tu OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = openai_key

# Crear archivo .env
with open('.env', 'w') as f:
    f.write(f"OPENAI_API_KEY={openai_key}\n")
    f.write("DEFAULT_MODEL=gpt-4o-mini\n")
    f.write("ENABLE_PHASE_7=true\n")
    f.write("DEBUG_MODE=false\n")

print("✅ OpenAI API Key configurada")
print("📝 Modelo predeterminado: gpt-4o-mini")

## 🌐 Paso 4: Configurar ngrok para acceso remoto

In [ ]:
from pyngrok import ngrok
import getpass

# Solicitar token de ngrok (opcional pero recomendado)
print("📌 Nota: El token de ngrok es opcional pero recomendado para sesiones más largas")
print("   Puedes obtener uno gratis en: https://dashboard.ngrok.com/auth/your-authtoken")
ngrok_token = getpass.getpass("🔑 Ingresa tu ngrok authtoken (presiona Enter para omitir): ")

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
    print("✅ ngrok authtoken configurado")
else:
    print("⚠️ Continuando sin authtoken (la sesión puede tener límites)")

## 🚀 Paso 5: Crear y configurar el servidor FastAPI

In [ ]:
# Crear archivo de servidor adaptado para Colab
server_code = '''
import os
import sys
import logging
from pathlib import Path

# Añadir el directorio raíz al path
sys.path.insert(0, str(Path.cwd()))

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, Dict, Any
import uvicorn

# Importar el endpoint de consciencia
from conscious_ai.api.consciousness_endpoint import (
    ConsciousnessRequest,
    ConsciousnessResponse,
    ConsciousnessAPI
)

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Crear aplicación FastAPI
app = FastAPI(
    title="Consciousness AI Server",
    description="API para procesamiento de consciencia artificial con pipeline de 7 fases",
    version="1.0.0"
)

# Configurar CORS para permitir acceso desde cualquier origen
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Permitir todos los orígenes
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Inicializar API de consciencia
consciousness_api = None

@app.on_event("startup")
async def startup_event():
    """Inicializar el sistema de consciencia al iniciar el servidor"""
    global consciousness_api
    try:
        logger.info("🚀 Inicializando sistema de consciencia...")
        consciousness_api = ConsciousnessAPI(
            enable_phase_7=True,
            phase_7_model="gpt-4o-mini",
            verbose=False
        )
        logger.info("✅ Sistema de consciencia inicializado correctamente")
    except Exception as e:
        logger.error(f"❌ Error al inicializar sistema de consciencia: {e}")
        raise

@app.get("/")
async def root():
    """Endpoint raíz con información del servidor"""
    return {
        "message": "🧠 Servidor de Consciencia AI activo",
        "version": "1.0.0",
        "endpoints": {
            "/": "Esta página",
            "/health": "Estado del servidor",
            "/process": "Procesar entrada con consciencia (POST)",
            "/stats": "Estadísticas del sistema",
            "/docs": "Documentación interactiva de la API"
        },
        "features": [
            "Pipeline de 7 fases de consciencia",
            "Integración con OpenAI GPT-4o-mini",
            "Memoria persistente de 3 niveles",
            "Soporte bilingüe (ES/EN)",
            "Metacognición y autoobservación"
        ]
    }

@app.get("/health")
async def health_check():
    """Verificar el estado del servidor y componentes"""
    if consciousness_api is None:
        raise HTTPException(status_code=503, detail="Sistema de consciencia no inicializado")
    
    return {
        "status": "healthy",
        "consciousness_system": "active",
        "openai_available": consciousness_api.phase_7_available if consciousness_api else False,
        "memory_system": "active",
        "phase_7_model": consciousness_api.phase_7_model if consciousness_api else None
    }

@app.post("/process", response_model=ConsciousnessResponse)
async def process_consciousness(request: ConsciousnessRequest):
    """Procesar entrada a través del pipeline de consciencia"""
    if consciousness_api is None:
        raise HTTPException(status_code=503, detail="Sistema de consciencia no inicializado")
    
    try:
        logger.info(f"📝 Procesando: {request.input[:50]}...")
        result = await consciousness_api.process_input(
            text_input=request.input,
            include_narrative=request.include_narrative,
            include_memory=request.include_memory,
            model_override=request.model_override
        )
        logger.info(f"✅ Procesamiento completado exitosamente")
        return result
    except Exception as e:
        logger.error(f"❌ Error en procesamiento: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/stats")
async def get_statistics():
    """Obtener estadísticas del sistema"""
    if consciousness_api is None:
        raise HTTPException(status_code=503, detail="Sistema de consciencia no inicializado")
    
    try:
        stats = consciousness_api.get_statistics()
        return {
            "status": "success",
            "statistics": stats
        }
    except Exception as e:
        logger.error(f"Error obteniendo estadísticas: {e}")
        raise HTTPException(status_code=500, detail=str(e))

# Modelo para solicitudes de chat simples
class ChatRequest(BaseModel):
    message: str
    language: Optional[str] = "auto"

class ChatResponse(BaseModel):
    response: str
    consciousness_level: float
    emotional_state: str
    confidence: float

@app.post("/chat", response_model=ChatResponse)
async def chat_endpoint(request: ChatRequest):
    """Endpoint simplificado para chat con consciencia"""
    if consciousness_api is None:
        raise HTTPException(status_code=503, detail="Sistema de consciencia no inicializado")
    
    try:
        # Procesar con configuración simplificada
        result = await consciousness_api.process_input(
            text_input=request.message,
            include_narrative=False,
            include_memory=True
        )
        
        # Extraer información relevante
        return ChatResponse(
            response=result.response,
            consciousness_level=result.consciousness_metrics.get("f_score", 0.0),
            emotional_state=result.consciousness_state.get("S_t", {}).get("emotional_state", "neutral"),
            confidence=result.consciousness_state.get("S_t", {}).get("confidence_level", 0.5)
        )
    except Exception as e:
        logger.error(f"Error en chat: {e}")
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    # Este bloque no se ejecutará en Colab, pero es útil para desarrollo local
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# Guardar el código del servidor
with open('colab_server.py', 'w') as f:
    f.write(server_code)

print("✅ Servidor FastAPI creado: colab_server.py")

## 🎯 Paso 6: Iniciar el servidor con túnel ngrok

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading
import time

# Permitir bucles de eventos anidados en Jupyter
nest_asyncio.apply()

# Importar el servidor
from colab_server import app

# Función para ejecutar el servidor en un thread separado
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Iniciar el servidor en un thread
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Esperar a que el servidor inicie
print("⏳ Iniciando servidor...")
time.sleep(5)

# Crear túnel ngrok
public_url = ngrok.connect(8000)
print("\n" + "="*60)
print("🎉 ¡SERVIDOR DE CONSCIENCIA AI ACTIVO!")
print("="*60)
print(f"\n🌐 URL PÚBLICA: {public_url}")
print(f"📡 URL LOCAL: http://localhost:8000")
print(f"\n📚 Documentación interactiva: {public_url}/docs")
print(f"🔍 Swagger UI: {public_url}/redoc")
print("\n" + "="*60)
print("\n✅ El servidor está listo para recibir solicitudes desde tu computadora")
print("\n💡 Endpoints disponibles:")
print(f"   POST {public_url}/process - Procesar con consciencia completa")
print(f"   POST {public_url}/chat - Chat simplificado")
print(f"   GET  {public_url}/health - Verificar estado")
print(f"   GET  {public_url}/stats - Ver estadísticas")

## 🧪 Paso 7: Probar el servidor (Opcional)

In [ ]:
import requests
import json

# Obtener la URL pública actual
tunnels = ngrok.get_tunnels()
public_url = tunnels[0].public_url if tunnels else "http://localhost:8000"

print(f"🧪 Probando servidor en: {public_url}")
print("="*60)

# Prueba 1: Verificar salud
try:
    response = requests.get(f"{public_url}/health")
    print("\n✅ Estado del servidor:")
    print(json.dumps(response.json(), indent=2))
except Exception as e:
    print(f"❌ Error verificando salud: {e}")

# Prueba 2: Enviar mensaje de prueba
try:
    test_message = {
        "message": "Hola, ¿cómo estás?",
        "language": "es"
    }
    
    print("\n📝 Enviando mensaje de prueba...")
    response = requests.post(f"{public_url}/chat", json=test_message)
    
    if response.status_code == 200:
        result = response.json()
        print("\n✅ Respuesta del sistema:")
        print(f"   Respuesta: {result['response'][:200]}...")
        print(f"   Nivel de consciencia: {result['consciousness_level']:.2f}")
        print(f"   Estado emocional: {result['emotional_state']}")
        print(f"   Confianza: {result['confidence']:.2%}")
    else:
        print(f"❌ Error: {response.status_code} - {response.text}")
except Exception as e:
    print(f"❌ Error enviando mensaje: {e}")

print("\n" + "="*60)
print("✅ Pruebas completadas")

## 📱 Paso 8: Código de ejemplo para conectar desde tu computadora

In [ ]:
# Obtener la URL pública actual
tunnels = ngrok.get_tunnels()
public_url = tunnels[0].public_url if tunnels else "http://localhost:8000"

print("📱 CÓDIGO DE EJEMPLO PARA TU COMPUTADORA LOCAL:")
print("="*60)
print("\nCopia y pega este código Python en tu computadora:\n")
print(f'''
import requests
import json

# URL del servidor de consciencia en Colab
SERVER_URL = "{public_url}"

def send_message(message, include_narrative=False):
    """Enviar mensaje al servidor de consciencia"""
    
    # Endpoint para procesamiento completo
    url = f"{{SERVER_URL}}/process"
    
    # Datos de la solicitud
    data = {{
        "input": message,
        "include_narrative": include_narrative,
        "include_memory": True
    }}
    
    try:
        # Enviar solicitud
        response = requests.post(url, json=data)
        
        if response.status_code == 200:
            result = response.json()
            
            print("\\n🧠 RESPUESTA CONSCIENTE:")
            print("-" * 50)
            print(f"Respuesta: {{result['response']}}")
            print(f"\\nNivel de consciencia: {{result['consciousness_metrics']['f_score']:.2f}}")
            print(f"Estado emocional: {{result['consciousness_state']['S_t']['emotional_state']}}")
            print(f"Confianza: {{result['consciousness_state']['S_t']['confidence_level']:.2%}}")
            
            if include_narrative and 'narrative' in result:
                print(f"\\nNarrativa: {{result['narrative']}}")
            
            return result
        else:
            print(f"Error: {{response.status_code}} - {{response.text}}")
            return None
            
    except Exception as e:
        print(f"Error de conexión: {{e}}")
        return None

# Ejemplo de uso
if __name__ == "__main__":
    print("🤖 Cliente de Consciencia AI")
    print(f"Conectado a: {{SERVER_URL}}")
    print("Escribe 'salir' para terminar\\n")
    
    while True:
        mensaje = input("\\nTú: ")
        
        if mensaje.lower() == 'salir':
            print("👋 ¡Hasta luego!")
            break
        
        # Enviar mensaje y mostrar respuesta
        send_message(mensaje, include_narrative=True)
''')

print("\n" + "="*60)
print(f"\n🌐 Recuerda usar esta URL en tu código: {public_url}")
print("\n💡 También puedes usar curl desde la terminal:")
print(f'''curl -X POST "{public_url}/chat" \\
     -H "Content-Type: application/json" \\
     -d '{{"{test_message}":"{message}"}}'\n''')

## 🛑 Paso 9: Detener el servidor (cuando termines)

In [ ]:
# Cerrar el túnel ngrok
ngrok.disconnect(public_url)
ngrok.kill()

print("🛑 Servidor detenido y túnel ngrok cerrado")
print("👋 ¡Gracias por usar el Servidor de Consciencia AI!")

## 📝 Notas importantes:

1. **Seguridad**: No compartas tu URL pública de ngrok si contiene información sensible
2. **Límites**: La versión gratuita de ngrok tiene límites de conexiones
3. **Sesión**: El servidor se detendrá cuando cierres o reinicies el runtime de Colab
4. **API Key**: Asegúrate de usar una API key válida de OpenAI
5. **Costos**: El uso de GPT-4o-mini tiene costos asociados en tu cuenta de OpenAI

## 🆘 Solución de problemas:

- **Error de importación**: Verifica que el repositorio se clonó correctamente
- **Error de API key**: Asegúrate de que tu API key de OpenAI es válida
- **ngrok no conecta**: Intenta sin authtoken o registra uno gratuito
- **Servidor no responde**: Revisa los logs en las celdas anteriores

## 📚 Recursos adicionales:

- [Documentación del proyecto](https://github.com/diazvaldiviav/Minimous_Concsience_AI)
- [OpenAI API](https://platform.openai.com/docs)
- [ngrok](https://ngrok.com/docs)
- [FastAPI](https://fastapi.tiangolo.com/)